### **Step 1 — Input & Split into Characters**

In [1]:
# Input
word = "Engineering"

# Split into characters and assign token id
chars = list(word)
token_ids = list(range(len(chars)))

print("Characters:", chars)
print("Token IDs :", token_ids)

Characters: ['E', 'n', 'g', 'i', 'n', 'e', 'e', 'r', 'i', 'n', 'g']
Token IDs : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


### **Step 2 — Get Unique Characters & Build vocab.json**

In [2]:
# Get unique characters and assign unique id
vocab = {}
uid = 0
for char in chars:
    if char not in vocab:
        vocab[char] = uid
        uid += 1

print("vocab.json:", vocab)

vocab.json: {'E': 0, 'n': 1, 'g': 2, 'i': 3, 'e': 4, 'r': 5}


### **Step 3 — Get All Adjacent Pairs**

In [3]:
# Make pairs of adjacent tokens
def get_pairs(tokens):
    pairs = []
    for i in range(len(tokens) - 1):
        pairs.append((tokens[i], tokens[i+1]))
    return pairs

tokens = chars.copy()  # working list of tokens

pairs = get_pairs(tokens)
print("All pairs:")
for p in pairs:
    print(" ", p)

All pairs:
  ('E', 'n')
  ('n', 'g')
  ('g', 'i')
  ('i', 'n')
  ('n', 'e')
  ('e', 'e')
  ('e', 'r')
  ('r', 'i')
  ('i', 'n')
  ('n', 'g')


### **Step 4 — Find Repeating or Most Frequent Pair**

In [4]:
# Check for repeating pair first
# If no repeating pair, pick most frequent pair
from collections import Counter

def find_best_pair(tokens):
    pairs = get_pairs(tokens)

    # Count frequency of each pair
    freq = Counter(pairs)

    # Check if any pair repeats (freq > 1)
    repeating = {p: f for p, f in freq.items() if f > 1}

    if repeating:
        best = max(repeating, key=repeating.get)
        print(f"  Repeating pair found: {best}")
    else:
        # No repeating — pick any (first one in your approach)
        best = pairs[0]
        print(f"  No repeat — picking pair: {best}")

    return best

### **Step 5 — Merge the Best Pair**

In [5]:
# Merge the chosen pair in token list
def merge_pair(tokens, pair):
    merged = pair[0] + pair[1]   # e.g. 'n'+'g' = 'ng'
    new_tokens = []
    i = 0
    while i < len(tokens):
        # If current and next match the pair, merge
        if i < len(tokens)-1 and tokens[i] == pair[0] and tokens[i+1] == pair[1]:
            new_tokens.append(merged)
            i += 2              # skip both
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

### **Step 6 — Run All Iterations**

In [6]:
# Full BPE loop — your approach
word = "Engineering"
tokens = list(word)

# Build initial vocab
vocab = {}
uid = 0
for ch in tokens:
    if ch not in vocab:
        vocab[ch] = uid
        uid += 1

print("=== Initial ===")
print("Tokens :", tokens)
print("Vocab  :", vocab)
print()

# Run iterations
for iteration in range(1, 20):
    pairs = get_pairs(tokens)
    if not pairs:
        break

    print(f"=== Iteration {iteration} ===")
    print("Pairs  :", pairs)

    # Find best pair
    best = find_best_pair(tokens)
    merged = best[0] + best[1]

    # Merge
    tokens = merge_pair(tokens, best)

    # Add to vocab
    if merged not in vocab:
        vocab[merged] = uid
        uid += 1

    print(f"  Merge : {best[0]} + {best[1]} = {merged}")
    print(f"  Tokens: {tokens}")
    print(f"  Vocab : {vocab}")
    print()

    # Stop if single token
    if len(tokens) == 1:
        print("=== Done! Single token reached ===")
        break

=== Initial ===
Tokens : ['E', 'n', 'g', 'i', 'n', 'e', 'e', 'r', 'i', 'n', 'g']
Vocab  : {'E': 0, 'n': 1, 'g': 2, 'i': 3, 'e': 4, 'r': 5}

=== Iteration 1 ===
Pairs  : [('E', 'n'), ('n', 'g'), ('g', 'i'), ('i', 'n'), ('n', 'e'), ('e', 'e'), ('e', 'r'), ('r', 'i'), ('i', 'n'), ('n', 'g')]
  Repeating pair found: ('n', 'g')
  Merge : n + g = ng
  Tokens: ['E', 'ng', 'i', 'n', 'e', 'e', 'r', 'i', 'ng']
  Vocab : {'E': 0, 'n': 1, 'g': 2, 'i': 3, 'e': 4, 'r': 5, 'ng': 6}

=== Iteration 2 ===
Pairs  : [('E', 'ng'), ('ng', 'i'), ('i', 'n'), ('n', 'e'), ('e', 'e'), ('e', 'r'), ('r', 'i'), ('i', 'ng')]
  No repeat — picking pair: ('E', 'ng')
  Merge : E + ng = Eng
  Tokens: ['Eng', 'i', 'n', 'e', 'e', 'r', 'i', 'ng']
  Vocab : {'E': 0, 'n': 1, 'g': 2, 'i': 3, 'e': 4, 'r': 5, 'ng': 6, 'Eng': 7}

=== Iteration 3 ===
Pairs  : [('Eng', 'i'), ('i', 'n'), ('n', 'e'), ('e', 'e'), ('e', 'r'), ('r', 'i'), ('i', 'ng')]
  No repeat — picking pair: ('Eng', 'i')
  Merge : Eng + i = Engi
  Tokens: ['Engi', 